# V24.7 route-90 commitment plus ILP shadow

Frozen full-sequence GPU audit of the complete `6bba + components + greedy` route cohort: 16 retrospective V24.3 regressions and 74 route-matched wins.

This notebook performs inference and read-only shadow evaluation. It does not train, tune thresholds, mutate production graphs, select between V19 and V24, generate a submission, or authorize deployment.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

BRANCH = "v24-score-first-tracking"
EXPECTED_COMMIT = "da98e99b542b6f49a0ed95ca7b40eccbe7f3820d"
ROOT = Path("/tmp/Atabey")

if not ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "https://github.com/drosadocastro-bit/Atabey.git",
            str(ROOT),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)

subprocess.run(
    ["git", "-C", str(ROOT), "checkout", "--detach", EXPECTED_COMMIT],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = f"{ROOT}:{ROOT / 'src'}:{ROOT / 'scripts'}"
print("Atabey commit verified:", actual_commit)

In [ ]:
pinned_official_packages = [
    "git+https://github.com/royerlab/tracksdata.git@39dccf3a243e44274759468cb31b2ad9e7fc1d09",
    "git+https://github.com/royerlab/kaggle-cell-tracking-competition.git@075fc5f5a52d11077f9dc2b074644618f26939e2",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pinned_official_packages],
    check=True,
)

official_runtime_packages = [
    "bidict>=0.23.1", "blosc2", "dask", "geff>=1.1.3.1.1",
    "ilpy>=0.5.1", "imagecodecs", "numba", "numcodecs>=0.13",
    "numpy==2.2.6", "scipy==1.16.3", "polars>=1.36.0",
    "psygnal>=0.14.0", "pyarrow", "rich", "rustworkx>=0.17.1",
    "scikit-image>=0.24.0", "sqlalchemy>=2", "tqdm", "typing-extensions",
    "zarr>=3.0.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *official_runtime_packages],
    check=True,
)

runtime_probe_code = """
import json
import numpy as np
import scipy
from scipy.optimize import milp
import torch

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before route-90 execution"
print(json.dumps({
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "milp": milp.__name__,
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0),
}, sort_keys=True))
"""
runtime_probe = subprocess.run(
    [sys.executable, "-c", runtime_probe_code],
    check=True,
    capture_output=True,
    text=True,
    env=RUN_ENV,
)
print("Python:", sys.version.split()[0])
print("Runtime probe:", runtime_probe.stdout.strip())

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
EXPECTED_CHECKPOINT_SHA256 = (
    "02e1d65756c3dc5928f68a66a8b0ef99be2a6905fa7bc017aa1d87dbe632fd03"
)
EXPECTED_PREDICTOR_SHA256 = (
    "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9"
)

train_candidates = []
for pattern in ("*/train", "*/*/train", "*/*/*/train"):
    for candidate in INPUT_ROOT.glob(pattern):
        if (
            candidate.is_dir()
            and len(list(candidate.glob("*.zarr"))) == 199
            and len(list(candidate.glob("*.geff"))) == 199
        ):
            train_candidates.append(candidate)
train_candidates = sorted(set(train_candidates))
assert len(train_candidates) == 1, f"Expected one 199-sample train directory: {train_candidates}"
TRAIN_DIR = train_candidates[0]

auxiliary_roots = sorted(
    root
    for root in INPUT_ROOT.iterdir()
    if root.is_dir() and not TRAIN_DIR.is_relative_to(root)
)
assert auxiliary_roots, "No auxiliary Kaggle input roots found"

predictor_candidates = []
for root in auxiliary_roots:
    predictor_candidates.extend(root.rglob("predict_unet_transformer.py"))
predictor_candidates = sorted(
    path for path in set(predictor_candidates) if path.parent.name == "scripts"
)
matching_predictors = [
    path
    for path in predictor_candidates
    if hashlib.sha256(path.read_bytes()).hexdigest() == EXPECTED_PREDICTOR_SHA256
]
assert len(matching_predictors) == 1, f"Expected one exact support predictor: {matching_predictors}"
SUPPORT_REPO = matching_predictors[0].parents[1]

weight_candidates = []
for root in auxiliary_roots:
    weight_candidates.extend(root.rglob("edge_predictor_best.pth"))
matching_weights = [
    candidate
    for candidate in sorted(set(weight_candidates))
    if (candidate.parent / "config.json").exists()
    and hashlib.sha256(candidate.read_bytes()).hexdigest() == EXPECTED_CHECKPOINT_SHA256
]
assert len(matching_weights) == 1, f"Expected one exact frozen checkpoint: {matching_weights}"
WEIGHTS = matching_weights[0]

checkpoint_config = json.loads((WEIGHTS.parent / "config.json").read_text(encoding="utf-8"))
assert checkpoint_config["window_size"] == 2
assert checkpoint_config["downsample"] == [1, 4, 4]
assert checkpoint_config["unet_out_channels"] == 32
assert checkpoint_config["pool_kernel_um"] == 5.0

print("Train:", TRAIN_DIR)
print("Support:", SUPPORT_REPO)
print("Checkpoint:", WEIGHTS)

In [ ]:
focused_tests = [
    ROOT / "tests/test_v24_7_route_90_shadow.py",
    ROOT / "tests/test_commitment_ilp_shadow.py",
    ROOT / "tests/test_commitment_shadow.py",
    ROOT / "tests/test_bounded_ilp_shadow.py",
]
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", *map(str, focused_tests)],
    check=True,
    env=RUN_ENV,
)
print("Route-90, commitment, and ILP shadow tests passed")

## Execution gate

This run is authorized only for the frozen 90-sample shadow audit. Do not change the cohort, thresholds, time horizon, intervention penalty, or solver budgets. A completed run remains retrospective evidence and cannot authorize submission or production mutation.

In [ ]:
AUTHORIZE_ROUTE_90_SHADOW = True
assert AUTHORIZE_ROUTE_90_SHADOW is True

CONTRACT_PATH = ROOT / "tests/fixtures/v24_7_route_90_shadow.json"
PREREGISTRATION_PATH = ROOT / "V24_7_ROUTE_90_GPU_SHADOW_PREREGISTRATION.md"
contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
assert contract["cohort"]["expected_samples"] == 90
assert len(contract["cohort"]["sample_ids"]) == 90
assert len(contract["cohort"]["regression_sample_ids"]) == 16
assert contract["execution"]["max_timepoints"] is None
assert contract["execution"]["unet_batch_size"] == 4
assert all(value is False for value in contract["boundaries"].values())

OUTPUT_DIR = Path("/kaggle/working/v24_7_route_90_shadow")
print("Cohort: 90 route-matched samples")
print("Retrospective regressions: 16")
print("Route controls: 74")
print("Output:", OUTPUT_DIR)

In [ ]:
command = [
    sys.executable,
    "-u",
    str(ROOT / "scripts/run_v24_7_route_90_shadow.py"),
    "--train-dir", str(TRAIN_DIR),
    "--support-repo", str(SUPPORT_REPO),
    "--weights", str(WEIGHTS),
    "--shadow-contract", str(CONTRACT_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--unet-batch-size", "4",
    "--resume",
    "--verify-determinism",
]
started = time.time()
subprocess.run(command, check=True, env=RUN_ENV)
elapsed_seconds = time.time() - started
print("Route-90 elapsed hours:", elapsed_seconds / 3600.0)

In [ ]:
summary = json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
assert summary["status"] == "v24_7_route_90_shadow_complete"
assert summary["sample_count"] == 90
assert summary["expected_sample_count"] == 90
assert summary["complete_cohort"] is True
assert summary["determinism_verified"] is True
assert summary["assignment_enabled"] is False
assert summary["selector_enabled"] is False
assert summary["production_graph_mutation"] is False
assert summary["submission_authorized"] is False
assert summary["threshold_tuning"] is False
assert len(list((OUTPUT_DIR / "samples").glob("*.json"))) == 90

print("Status:", summary["status"])
print("Aggregate:")
print(json.dumps(summary["aggregate"], indent=2, sort_keys=True))

In [ ]:
BUNDLE = Path("/kaggle/working/v24_7_route_90_shadow_outputs")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
shutil.copytree(OUTPUT_DIR, BUNDLE / "run")
shutil.copy2(CONTRACT_PATH, BUNDLE / CONTRACT_PATH.name)
shutil.copy2(PREREGISTRATION_PATH, BUNDLE / PREREGISTRATION_PATH.name)

run_record = {
    "mode": "v24_7_route_90_commitment_ilp_shadow",
    "atabey_commit": EXPECTED_COMMIT,
    "checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256,
    "predictor_sha256": EXPECTED_PREDICTOR_SHA256,
    "sample_count": summary["sample_count"],
    "elapsed_seconds": elapsed_seconds,
    "status": summary["status"],
    "no_training": True,
    "selector_enabled": False,
    "submission_authorized": False,
}
(BUNDLE / "notebook_run_record.json").write_text(
    json.dumps(run_record, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
archive = shutil.make_archive(str(BUNDLE), "zip", BUNDLE)
print("Download:", archive)

## Interpretation boundary

This route-90 cohort is retrospective: all 199 labels were already opened. Report raw persistent/reconverging denominators, proposal classes, intervention sizes, solver failures, and official counterfactual deltas. Positive zero-penalty results do not authorize lowering the containment penalty. Add-only gains do not count as ownership-rewrite evidence. No result from this notebook authorizes training, threshold tuning, automatic selection, production graph mutation, or submission.